# `pipeline.ipynb` - Orquestador del pipeline completo

Une todos los módulos anteriores dentro de un `try/except` general: extracción (distritos + tabla socioeconómica + puntos de recarga), geoprocesamiento (spatial join + agregación), cálculo de KPIs (densidad, normalización, IOI) y generación del visor cartográfico. Si cualquier paso falla, se registra en el log con traza completa (`exc_info=True`) y se relanza la excepción.

> Depende de **todos** los módulos anteriores, así que debe ser el último `%run` de la cadena en `main.ipynb`.

In [ ]:
import logging
from datetime import datetime

## Orquestador

In [ ]:
def ejecutar_pipeline():
    """
    Ejecuta el pipeline GeoKW completo de principio a fin: extraccion,
    geoprocesamiento, calculo de KPIs y generacion del mapa. Devuelve
    (master, df_validos_estaciones, df_cuarentena_estaciones, origen_estaciones).
    """
    fecha_inicio = datetime.now()
    logger.info("=" * 70)
    logger.info("INICIO DE EJECUCION DEL PIPELINE ETL - PROYECTO GEOKW")
    logger.info("=" * 70)

    try:
        # 1) Extraccion: distritos (IDE Sevilla) y tabla socioeconomica (data seeding)
        gdf_distritos = extraer_distritos()
        df_socio = extraer_tabla_socioeconomica()

        # 2) Extraccion resiliente: puntos de recarga (Open Charge Map + fallback)
        estaciones, origen_estaciones = obtener_estaciones_carga()
        logger.info(f"Origen de los puntos de recarga: {origen_estaciones} ({len(estaciones)} puntos)")

        # 3) Geoprocesamiento: spatial join + agregacion por distrito
        gdf_estaciones = estaciones_a_geodataframe(estaciones)
        df_validos, df_cuarentena = cruzar_estaciones_con_distritos(gdf_estaciones, gdf_distritos)
        df_potencia = agregar_potencia_por_distrito(df_validos, sorted(POBLACION_DISTRITOS.keys()))

        # 4) Calculo de KPIs: densidad energetica, normalizacion min-max, IOI
        master = construir_tabla_maestra(gdf_distritos, df_socio, df_potencia)
        master = calcular_densidad_energetica(master)
        master = calcular_ioi(master)

        # 5) Generacion del visor cartografico
        generar_mapa(master, df_validos, df_cuarentena, RUTA_SALIDA_MAPA)

        fecha_fin = datetime.now()
        duracion = (fecha_fin - fecha_inicio).total_seconds()
        logger.info(
            f"EJECUCION FINALIZADA CORRECTAMENTE. Distritos: {len(master)} | "
            f"Puntos validos: {len(df_validos)} | Puntos en cuarentena: {len(df_cuarentena)} | "
            f"Origen demografia/potencia: {origen_estaciones} | Duracion: {duracion:.2f} s"
        )
        logger.info("=" * 70)
        return master, df_validos, df_cuarentena, origen_estaciones

    except Exception as error:
        logger.error(f"EJECUCION FINALIZADA CON ERROR: {error}", exc_info=True)
        raise

---
✅ **Orquestador definido.** Se ejecuta y se verifica en `main.ipynb`.